In [0]:
# Path to your Volume
path = "/Volumes/workspace/default/olist_files"

# Loading the necessary tables
df_customers = spark.read.csv(f"{path}/olist_customers_dataset.csv", header=True, inferSchema=True)
df_orders = spark.read.csv(f"{path}/olist_orders_dataset.csv", header=True, inferSchema=True)
df_payments = spark.read.csv(f"{path}/olist_order_payments_dataset.csv", header=True, inferSchema=True) 

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import sum, col, rank

# 1. Join tables and aggregate spend per customer/city
customer_spend = df_payments.join(df_orders, "order_id") \
    .join(df_customers, "customer_id") \
    .groupBy("customer_city", "customer_id") \
    .agg(sum("payment_value").alias("total_spend"))

# 2. Define the Window: Partition by City, Order by Spend (Highest first)
window_spec = Window.partitionBy("customer_city").orderBy(col("total_spend").desc())

# 3. Apply the rank and filter for the top 3
task_1_result = customer_spend.withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") <= 3)

# View the output
display(task_1_result)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import sum, col, to_date

path = "/Volumes/workspace/default/olist_files"

# Load datasets
df_orders = spark.read.csv(f"{path}/olist_orders_dataset.csv", header=True, inferSchema=True)
df_payments = spark.read.csv(f"{path}/olist_order_payments_dataset.csv", header=True, inferSchema=True)

# Join and extract the date
sales_data = df_payments.join(df_orders, "order_id") \
    .withColumn("order_date", to_date(col("order_purchase_timestamp")))

In [0]:
# 1. Aggregate total sales per day
daily_sales_df = sales_data.groupBy("order_date") \
    .agg(sum("payment_value").alias("daily_sales")) \
    .orderBy("order_date")

# 2. Define Window: Ordered by date, ranging from the start of time to the current row
window_spec = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 3. Calculate running total
task_2_result = daily_sales_df.withColumn("running_total", sum("daily_sales").over(window_spec))

display(task_2_result)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import sum, col, dense_rank

path = "/Volumes/workspace/default/olist_files"

# Load datasets
df_items = spark.read.csv(f"{path}/olist_order_items_dataset.csv", header=True, inferSchema=True)
df_products = spark.read.csv(f"{path}/olist_products_dataset.csv", header=True, inferSchema=True)
df_translation = spark.read.csv(f"{path}/product_category_name_translation.csv", header=True, inferSchema=True)

In [0]:
# 1. Join items with products and translations
product_sales = df_items.join(df_products, "product_id") \
    .join(df_translation, "product_category_name")

# 2. Calculate total sales per product and category
category_revenue = product_sales.groupBy("product_category_name_english", "product_id") \
    .agg(sum("price").alias("total_sales"))

# 3. Define Window: Partition by Category, Order by Sales Descending
window_spec = Window.partitionBy("product_category_name_english").orderBy(col("total_sales").desc())

# 4. Apply DENSE_RANK (as requested in the task)
task_3_result = category_revenue.withColumn("rank", dense_rank().over(window_spec))

# Optionally filter for the #1 product in each category
top_products = task_3_result.filter(col("rank") == 1)

display(top_products)

In [0]:
from pyspark.sql.functions import sum, col

path = "/Volumes/workspace/default/olist_files"

# Load datasets
df_orders = spark.read.csv(f"{path}/olist_orders_dataset.csv", header=True, inferSchema=True)
df_payments = spark.read.csv(f"{path}/olist_order_payments_dataset.csv", header=True, inferSchema=True)

In [0]:
# 1. Join payments and orders on order_id
# 2. Group by customer_id and sum the payment_value
task_4_result = df_payments.join(df_orders, "order_id") \
    .groupBy("customer_id") \
    .agg(sum("payment_value").alias("total_spend"))

# View the result
display(task_4_result)

In [0]:
from pyspark.sql.functions import when, col, count

# Assuming 'task_4_result' is the DataFrame from the previous step containing 'customer_id' and 'total_spend'

# 1. Apply the segmentation rules
segmented_customers = task_4_result.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

# View the individual customer segments
display(segmented_customers)

In [0]:
# 2. Group by segment to see the count of customers in each tier
segment_counts = segmented_customers.groupBy("segment") \
    .agg(count("customer_id").alias("customer_count")) \
    .orderBy(col("customer_count").desc())

display(segment_counts)

In [0]:
from pyspark.sql.functions import count, col

# 1. Get order counts per customer
df_order_counts = df_orders.groupBy("customer_id") \
    .agg(count("order_id").alias("total_orders"))

# 2. Get customer city information (deduplicated to ensure 1 row per customer)
df_customer_geo = df_customers.select("customer_id", "customer_city").dropDuplicates(["customer_id"])

In [0]:
# Join the pieces together
final_reporting_table = segmented_customers \
    .join(df_customer_geo, "customer_id", "left") \
    .join(df_order_counts, "customer_id", "left") \
    .select(
        "customer_id", 
        "customer_city", 
        "total_spend", 
        "segment", 
        "total_orders"
    )

# View the final pipeline output
display(final_reporting_table)